# W6｜探索性資料分析 – Matplotlib
**智慧醫療與機器學習基礎**　臺北醫學大學 智慧醫療學士學位學程　授課教師：王三源　課程助教：古珉瑄（醫資所，m610115009@tmu.edu.tw）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/FIHML/blob/main/notebooks/W06_matplotlib.ipynb)

### 今天的情境
上週你用 pandas 做出了失智與非失智住民的比較表。主任看完說：
> 「表格裡睡眠時數兩組平均差不多，所以睡眠跟失智沒關係囉？」

護理長也來問：
> 「可以給我一張圖，讓我在交班會議上一眼看懂各機構的狀況嗎？」

今天要用圖回答這兩個問題。

> ⚠️ 本資料為**教學用模擬資料**，不對應任何真實個人。資料欄位說明見 [長照常用評估量表說明（PDF）](https://github.com/wang3yuan/FIHML/blob/main/docs/W05_長照評估量表說明.pdf)。

### 執行環境
本教材已在 **Python 3.12＋pandas 2.2.3／matplotlib 3.9.4** 與 **pandas 3.0.6／matplotlib 3.11.2** 兩組版本測試通過。執行下面第一格會印出你目前的版本。

### 使用方式
1. 「檔案 → 在雲端硬碟中儲存複本」，之後都在自己的複本上作業。
2. **由上往下依序執行**；看到 **▶ 實作** 的地方，請在 `# TODO` 處寫程式。

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print("Python 版本：", sys.version.split()[0])
print("pandas 版本：", pd.__version__)
print("matplotlib 版本：", matplotlib.__version__)

## 0. 設定中文字型（每次開啟 notebook 都要執行一次）
Colab 預設沒有中文字型，圖上的中文會變成「□□□」。下面這格會下載思源黑體（Noto Sans TC，開源授權 OFL）並設為預設字型。

In [ ]:
import urllib.request
import matplotlib.font_manager as fm

FONT_URL = "https://raw.githubusercontent.com/wang3yuan/FIHML/main/fonts/"
for name in ["NotoSansTC-Regular.ttf", "NotoSansTC-Bold.ttf"]:
    urllib.request.urlretrieve(FONT_URL + name, name)
    fm.fontManager.addfont(name)

plt.rcParams["font.family"] = "Noto Sans TC"
plt.rcParams["axes.unicode_minus"] = False   # 讓負號正常顯示
plt.rcParams["figure.dpi"] = 110

# 今天全程使用的配色：同一個群組永遠用同一個顏色
C_NO, C_YES = "#00897B", "#D06F1A"          # 非失智、失智
GROUP = {0: "非失智", 1: "失智"}

fig, ax = plt.subplots(figsize=(4, 1))
ax.text(0.5, 0.5, "中文字型設定完成 ✓", ha="center", va="center", fontsize=16)
ax.axis("off")
plt.show()

## 讀取資料
- `ltc_dementia_clean.csv`：上週清理好的住民資料（每人一列，最近一次評估）
- `ltc_assessments_clean.csv`：清理好的**所有**評估紀錄（每人 1–4 筆，用來看趨勢）

In [ ]:
BASE = "https://raw.githubusercontent.com/wang3yuan/FIHML/main/data/ltc/"
df = pd.read_csv(BASE + "ltc_dementia_clean.csv")
ass = pd.read_csv(BASE + "ltc_assessments_clean.csv", parse_dates=["assess_date"])

# 上週做過的分組，這裡再做一次
df["age_group"] = pd.cut(df["age"], bins=[59, 74, 84, 120], labels=["65–74", "75–84", "85+"])
df["adl_level"] = pd.cut(df["adl_barthel"], bins=[-1, 20, 60, 90, 99, 100],
                         labels=["完全依賴", "嚴重依賴", "中度依賴", "輕度依賴", "完全獨立"])
print(df.shape, ass.shape)
df.head()

---
# 第 1 堂｜第一張圖與「分布」
## 1.1 三行畫出第一張圖
`fig, ax = plt.subplots()` 建立一張**畫布（Figure）**和一個**座標區（Axes）**，之後所有的畫圖指令都對 `ax` 下。

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["age"])
plt.show()

這張圖能用，但讀的人不知道 x 軸是什麼、單位是什麼。**一張圖至少要有：標題、軸標籤（含單位）。**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(df["age"], bins=20, color=C_NO, edgecolor="white")
ax.set_title("住民年齡分布（n = 900）")
ax.set_xlabel("年齡（歲）")
ax.set_ylabel("住民數")
plt.show()

## 1.2 bins：直方圖要切幾格？
同一份資料，切太少看不出形狀，切太多變得破碎。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=False)
for ax, b in zip(axes, [5, 20, 60]):
    ax.hist(df["age"], bins=b, color=C_NO, edgecolor="white")
    ax.set_title(f"bins = {b}")
    ax.set_xlabel("年齡（歲）")
axes[0].set_ylabel("住民數")
plt.tight_layout()
plt.show()

## 1.3 長條圖：類別的數量
長條圖比的是「長度」，所以 **y 軸一定要從 0 開始**。

In [ ]:
counts = df["facility"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(counts.index, counts.values, color=C_NO)
ax.set_ylim(0, counts.max() * 1.12)                 # 上方留空間放數字
ax.set_title("各機構住民數")
ax.set_ylabel("住民數")
for x, v in zip(counts.index, counts.values):      # 在長條上標數字
    ax.text(x, v + 5, str(v), ha="center")
plt.show()

💡 pandas 也有捷徑：`counts.plot.bar()`。背後用的就是 matplotlib，學會 `ax` 的寫法，兩種都看得懂。

## 1.4 存成圖檔
放進報告或簡報時，用 `savefig` 存成高解析度圖檔，`bbox_inches="tight"` 會把多餘的白邊裁掉。

In [ ]:
fig.savefig("facility_counts.png", dpi=200, bbox_inches="tight")
print("已存檔：facility_counts.png（左側「📁 檔案」可下載）")

---
## ▶ 實作 6-1（約 15 分鐘）

**(a)** 畫出 `spmsq_errors`（SPMSQ 錯誤題數）的直方圖。提示：錯誤題數是 0–10 的整數，用 `bins=range(0, 12)` 讓每個整數一格。加上標題與軸標籤。
分布是對稱的，還是偏向某一邊？

In [ ]:
# TODO：請在這裡寫程式


**(b)** 畫出各巴氏量表依賴程度（`adl_level`）的住民數長條圖。依賴程度有順序，請依「完全依賴 → 完全獨立」排列（提示：`value_counts().sort_index()`）。

In [ ]:
# TODO：請在這裡寫程式


**(c)** 畫出 `sleep_hours`（夜間睡眠時數）的直方圖，試試 `bins=10` 與 `bins=40`，哪一個比較能看出形狀？

In [ ]:
# TODO：請在這裡寫程式


**(d)** 把 (a) 的圖存成 `spmsq_hist.png`（dpi=200）。

In [ ]:
# TODO：請在這裡寫程式


---
# 第 2 堂｜比較兩群人
## 2.1 疊加直方圖
兩組疊在一起時，用 `alpha`（透明度）讓重疊處看得見，並加上**圖例**。

In [ ]:
g0 = df[df["dementia"] == 0]
g1 = df[df["dementia"] == 1]

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.hist(g0["spmsq_errors"], bins=range(0, 12), alpha=0.6, color=C_NO, label="非失智", align="left")
ax.hist(g1["spmsq_errors"], bins=range(0, 12), alpha=0.6, color=C_YES, label="失智", align="left")
ax.set_title("SPMSQ 錯誤題數：失智 vs 非失智")
ax.set_xlabel("錯誤題數（題）")
ax.set_ylabel("住民數")
ax.legend()
plt.show()

## 2.2 箱形圖：一次看中位數、範圍與離群值
箱子中間的線是**中位數**，箱子上下是第 1、3 四分位數（中間 50% 的住民），鬚是一般範圍，外面的點是離群值。

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
bp = ax.boxplot([g0["adl_barthel"].dropna(), g1["adl_barthel"].dropna()], patch_artist=True,
                medianprops={"color": "black"})
for box, col in zip(bp["boxes"], [C_NO, C_YES]):
    box.set_facecolor(col)
    box.set_alpha(0.6)
ax.set_xticks([1, 2], ["非失智", "失智"])
ax.set_title("巴氏量表：失智 vs 非失智")
ax.set_ylabel("巴氏量表（0–100 分）")
plt.show()

## 2.3 主任的問題：睡眠跟失智真的沒關係嗎？
上週的比較表裡，睡眠時數兩組平均幾乎一樣。畫出來看看：

In [ ]:
print(df.groupby("dementia")["sleep_hours"].agg(["mean", "std"]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
bp = axes[0].boxplot([g0["sleep_hours"].dropna(), g1["sleep_hours"].dropna()], patch_artist=True,
                medianprops={"color": "black"})
for box, col in zip(bp["boxes"], [C_NO, C_YES]):
    box.set_facecolor(col)
    box.set_alpha(0.6)
axes[0].set_xticks([1, 2], ["非失智", "失智"])
axes[0].set_title("平均差不多，但分布寬很多")
axes[0].set_ylabel("夜間睡眠時數（小時）")

axes[1].hist(g0["sleep_hours"].dropna(), bins=25, alpha=0.6, color=C_NO, label="非失智", density=True)
axes[1].hist(g1["sleep_hours"].dropna(), bins=25, alpha=0.6, color=C_YES, label="失智", density=True)
axes[1].set_title("失智組：睡太少、睡太多的人都比較多")
axes[1].set_xlabel("夜間睡眠時數（小時）")
axes[1].set_ylabel("比例密度")
axes[1].legend()
plt.tight_layout()
plt.show()

換個角度：**依睡眠時數分組，看每組的失智比例**。

In [ ]:
df["sleep_group"] = pd.cut(df["sleep_hours"], bins=[3, 5, 6, 7, 8, 9, 12],
                           labels=["≤5", "5–6", "6–7", "7–8", "8–9", ">9"])
rate = df.groupby("sleep_group", observed=True)["dementia"].mean()

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(rate.index.astype(str), rate.values, marker="o", color=C_YES, linewidth=2)
ax.set_ylim(0, 1)
ax.set_title("睡太少、睡太多，失智比例都比較高（U 型）")
ax.set_xlabel("夜間睡眠時數（小時）")
ax.set_ylabel("失智比例")
ax.grid(axis="y", alpha=0.3)
plt.show()

🔍 **重點**：平均值會把「兩端」抵銷掉。**平均差不多 ≠ 沒有關係**，一定要把分布畫出來看。

## 2.4 散佈圖：兩個連續變數的關係

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for g, col in [(0, C_NO), (1, C_YES)]:
    sub = df[df["dementia"] == g]
    ax.scatter(sub["adl_barthel"], sub["iadl"], s=18, alpha=0.5, color=col, label=GROUP[g])
ax.set_title("巴氏量表 vs IADL")
ax.set_xlabel("巴氏量表（0–100 分）")
ax.set_ylabel("IADL（0–8 分）")
ax.legend()
plt.show()

💡 點重疊得很厲害時（IADL 只有 0–8 的整數），`alpha` 越小越能看出密集程度。

## 2.5 subplots：多張圖並排比較
`plt.subplots(列數, 欄數)` 一次建立多個座標區，用 `axes[0]`、`axes[1]`… 分別畫。`sharey=True` 讓它們共用 y 軸刻度，比較才公平。

---
## ▶ 實作 6-2（約 20 分鐘）

**(a)** 畫出 `iadl` 的箱形圖，比較失智與非失智兩組。記得兩組的顏色要和前面一致（`C_NO`、`C_YES`）。

In [ ]:
# TODO：請在這裡寫程式


**(b)** 用 `plt.subplots(1, 3)` 並排三張長條圖，分別比較兩組的平均 `spmsq_errors`、`adl_barthel`、`falls_past_year`。
提示：`means = df.groupby("dementia")[欄位].mean()`；三張圖的單位不同，所以**不要**用 `sharey=True`。

In [ ]:
# TODO：請在這裡寫程式


**(c)** 畫出各年齡組（`age_group`）的失智比例長條圖，y 軸範圍設為 0–1。

In [ ]:
# TODO：請在這裡寫程式


**(d)** 從 (a)–(c) 中挑一張圖，用一句話寫下你看到的發現。

> 你的發現：

---
# 第 3 堂｜關聯、趨勢，以及「給人看的圖」
## 3.1 相關係數熱圖
相關係數介於 −1 到 1：正值代表一起變大，負值代表一個變大另一個變小，0 代表沒有**直線**關係。

In [ ]:
num = ["age", "education_years", "spmsq_errors", "adl_barthel", "iadl", "gds15",
       "falls_past_year", "sleep_hours", "social_per_week", "n_medications", "dementia"]
corr = df[num].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num)), num, rotation=45, ha="right")
ax.set_yticks(range(len(num)), num)
for i in range(len(num)):
    for j in range(len(num)):
        r = corr.iloc[i, j]
        ax.text(j, i, f"{r:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(r) > 0.6 else "black")
fig.colorbar(im, ax=ax, label="相關係數")
ax.set_title("各變數的相關係數")
plt.tight_layout()
plt.show()

🔍 看最後一列 `dementia`：`spmsq_errors` 相關最強，`sleep_hours` 只有 0.05。
可是 2.3 才看到睡眠和失智有明顯的 **U 型**關係！**相關係數只抓得到直線關係**，U 型會被算成接近 0。

## 3.2 趨勢：同一位住民的多次評估
`ass` 裡每位住民有 1–4 次評估（每半年一次）。把每次評估換算成「距離最近一次評估幾個月」，就能畫出變化。

In [ ]:
ass["visit_back"] = ass.groupby("resident_id")["assess_date"].rank(ascending=False).astype(int) - 1
ass["months"] = -6 * ass["visit_back"]          # 0 = 最近一次，-6 = 半年前……
trend = ass.groupby(["months", "dementia"])["spmsq_errors"].mean().unstack()
trend

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for g, col in [(0, C_NO), (1, C_YES)]:
    ax.plot(trend.index, trend[g], marker="o", linewidth=2, color=col, label=GROUP[g])
ax.set_xticks([-18, -12, -6, 0], ["18 個月前", "12 個月前", "6 個月前", "最近一次"])
ax.set_ylim(0, 6)
ax.set_title("SPMSQ 錯誤題數的變化")
ax.set_ylabel("平均錯誤題數（題）")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.show()

⚠️ 越早的時間點，有資料的住民越少（只有評估過 4 次的人才有 18 個月前的資料），每個點代表的住民不完全相同。

## 3.3 會誤導的圖：截斷的 y 軸
同一份數字，y 軸從哪裡開始，給人的印象差很多。

In [ ]:
rate = df.groupby("facility")["dementia"].mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(rate.index, rate.values, color=C_YES)
axes[0].set_ylim(0.38, 0.48)
axes[0].set_title("錯誤示範：y 軸從 0.38 開始，A 看起來是別人的好幾倍")
axes[1].bar(rate.index, rate.values, color=C_YES)
axes[1].set_ylim(0, 1)
axes[1].set_title("正確示範：y 軸從 0 開始，差距其實不大")
for ax in axes:
    ax.set_ylabel("失智比例")
plt.tight_layout()
plt.show()

### 好圖檢查清單
1. **標題**說出重點，而不只是變數名稱
2. **軸標籤**有單位
3. 長條圖 **y 軸從 0 開始**
4. **兩個以上群組要有圖例**，同一群組全程同一個顏色
5. 不要用**雙 y 軸**（兩個刻度疊在同一張圖，容易製造假的關聯）
6. 不需要 3D、陰影、彩虹配色
7. 在圖下方用一句話寫出「讀者應該看到什麼」

## 3.4 給護理長的一頁式儀表板
把幾張圖排成一頁，就是最陽春的「儀表板」。期末報告要說明的「部署情境」（例如護理站儀表板），就是從這裡開始想。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

# 左上：各機構失智比例
rate = df.groupby("facility")["dementia"].mean()
axes[0, 0].bar(rate.index, rate.values, color=C_YES)
axes[0, 0].set_ylim(0, 1)
axes[0, 0].set_title("各機構失智比例")
for x, v in zip(rate.index, rate.values):
    axes[0, 0].text(x, v + 0.02, f"{v:.0%}", ha="center")

# 右上：依賴程度
lv = df["adl_level"].value_counts().sort_index()
axes[0, 1].barh(lv.index.astype(str), lv.values, color=C_NO)
axes[0, 1].invert_yaxis()
axes[0, 1].set_title("巴氏量表依賴程度（人）")

# 左下：認知變化趨勢
for g, col in [(0, C_NO), (1, C_YES)]:
    axes[1, 0].plot(trend.index, trend[g], marker="o", color=col, label=GROUP[g])
axes[1, 0].set_xticks([-18, -12, -6, 0], ["-18 月", "-12 月", "-6 月", "最近"])
axes[1, 0].set_title("SPMSQ 平均錯誤題數變化")
axes[1, 0].legend()

# 右下：跌倒次數
falls = df.groupby("dementia")["falls_past_year"].mean()
axes[1, 1].bar(["非失智", "失智"], falls.values, color=[C_NO, C_YES])
axes[1, 1].set_title("過去一年平均跌倒次數")

fig.suptitle("長照機構住民概況（模擬資料）", fontsize=16, fontweight="bold")
plt.tight_layout()
fig.savefig("dashboard.png", dpi=200, bbox_inches="tight")
plt.show()

---
## ▶ 實作 6-3（約 15 分鐘）：做一張你自己的儀表板
你是護理長的助理。從下面挑 **4 張**圖（也可以自己設計），用 `plt.subplots(2, 2)` 排成一頁並存成 `my_dashboard.png`：
- 各機構的住民數或失智比例
- 各年齡組的失智比例
- 失智 vs 非失智的 GDS-15（憂鬱）箱形圖
- 多重用藥（`n_medications >= 5`）住民的比例
- 睡眠時數分組的失智比例（U 型）
- SPMSQ 或巴氏量表的變化趨勢

**要求**：總標題、每張小圖的標題、需要的軸標籤與單位、兩組比較要有圖例且顏色一致。

In [ ]:
# TODO：請在這裡寫程式


---
## ▶ 連結你的專題：用一張圖證明痛點存在（約 10 分鐘）
期中報告的「問題背景」需要證據。延續上週的資料字典，替同一個照護議題設計**一張**最能說服人的圖（可以先畫在紙上）：

| 項目 | 你的設計 |
|---|---|
| 這張圖要回答的問題 | 例：夜班時段的跌倒是不是比較多？ |
| 圖的種類 | 例：長條圖 |
| x 軸／y 軸（含單位） | 例：時段／跌倒次數（次） |
| 要比較的群組（顏色） | 例：有無夜間如廁協助 |
| 需要資料字典裡的哪些欄位 | |
| 誰會看這張圖？ | 例：護理長、照服員 |

想一想：這些資料**現在拿得到嗎**？拿不到的話，期末報告要說明怎麼蒐集。

---
### 本週 Matplotlib 指令總整理
| 目的 | 指令 |
|---|---|
| 建立畫布 | `fig, ax = plt.subplots(figsize=(寬, 高))`、`plt.subplots(列, 欄)` |
| 直方圖 | `ax.hist(資料, bins=)` |
| 長條圖 | `ax.bar(x, 高度)`、`ax.barh(y, 長度)` |
| 箱形圖 | `ax.boxplot([組1, 組2], patch_artist=True,
                medianprops={"color": "black"})` |
| 折線圖 | `ax.plot(x, y, marker="o")` |
| 散佈圖 | `ax.scatter(x, y, alpha=)` |
| 熱圖 | `ax.imshow(矩陣, cmap="RdBu_r", vmin=-1, vmax=1)`、`fig.colorbar()` |
| 標題與標籤 | `ax.set_title()`、`ax.set_xlabel()`、`ax.set_ylabel()`、`fig.suptitle()` |
| 刻度與範圍 | `ax.set_xticks(位置, 標籤)`、`ax.set_ylim()` |
| 圖例與文字 | `ax.legend()`、`ax.text(x, y, "文字")` |
| 排版與存檔 | `plt.tight_layout()`、`fig.savefig("檔名.png", dpi=200, bbox_inches="tight")` |